# 01 - Evidence Gate 0: Study Boundary and Acquisition Ledger

This notebook creates the first reviewable study boundary for Phase 1.

It does not produce findings, scores, or intervention recommendations. Its purpose is to make the boundary and acquisition ledger clear enough for external review.

## Evidence Rule

The boundary generated here is provisional. It can be used to acquire and audit public data, but it cannot support funding-facing claims until reviewed and validated.

In [ ]:
from pathlib import Path
import json
import sys

root_candidates = (Path.cwd(), Path.cwd() / 'phase1_spinelens_ai', *Path.cwd().parents)
PROJECT_ROOT = next(path for path in root_candidates if (path / 'src' / 'spinelens').exists())
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

DATA = PROJECT_ROOT / 'data'
INTERIM = DATA / 'interim'
INTERIM.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from shapely.ops import unary_union

anchors = pd.read_csv(DATA / 'study_area_anchors_phase1.csv')
anchors

## Anchor Review

All coordinates must remain marked as provisional until reviewed. If an anchor is wrong, edit `data/study_area_anchors_phase1.csv` first, then re-run this notebook.

In [ ]:
required_columns = {
    'anchor_id',
    'anchor_name',
    'anchor_role',
    'latitude',
    'longitude',
    'coordinate_status',
    'validation_status',
}
missing = required_columns - set(anchors.columns)
assert not missing, f'Missing columns: {sorted(missing)}'
assert anchors['anchor_id'].is_unique, 'Anchor IDs must be unique.'
assert anchors['latitude'].between(52.0, 53.0).all(), 'Latitude values do not look like Birmingham coordinates.'
assert anchors['longitude'].between(-2.5, -1.0).all(), 'Longitude values do not look like Birmingham coordinates.'
assert (anchors['validation_status'] == 'not_validated').all(), 'Gate 0 expects unvalidated seed anchors.'

anchors[['anchor_id', 'anchor_name', 'anchor_role', 'coordinate_status', 'validation_status', 'validation_action']]

## Boundary Generation

Method:

1. Convert anchor points from WGS84 to British National Grid.
2. Create a convex hull around the anchors.
3. Buffer the hull by 500 metres.
4. Export the boundary as WGS84 GeoJSON.

The 500 metre buffer is a starting assumption for data acquisition. It should be reviewed before analysis.

In [ ]:
BOUNDARY_BUFFER_METERS = 500

anchor_points = [Point(lon, lat) for lon, lat in zip(anchors['longitude'], anchors['latitude'])]
anchors_gdf = gpd.GeoDataFrame(anchors, geometry=anchor_points, crs='EPSG:4326')
anchors_bng = anchors_gdf.to_crs('EPSG:27700')

hull_bng = unary_union(list(anchors_bng.geometry)).convex_hull
boundary_bng = hull_bng.buffer(BOUNDARY_BUFFER_METERS)

boundary = gpd.GeoDataFrame(
    [
        {
            'boundary_id': 'phase1_provisional_corridor_boundary_v0',
            'status': 'provisional_not_field_validated',
            'method': 'anchor_convex_hull_buffer',
            'buffer_meters': BOUNDARY_BUFFER_METERS,
            'working_crs': 'EPSG:27700',
            'output_crs': 'EPSG:4326',
            'funding_use': 'not_approved',
        }
    ],
    geometry=[boundary_bng],
    crs='EPSG:27700',
).to_crs('EPSG:4326')

boundary_path = INTERIM / 'study_area_boundary_phase1.geojson'
boundary.to_file(boundary_path, driver='GeoJSON')
boundary_path

In [ ]:
boundary_bng_area_sq_km = boundary_bng.area / 1_000_000
bounds_wgs84 = boundary.total_bounds.tolist()

metadata = {
    'boundary_id': 'phase1_provisional_corridor_boundary_v0',
    'status': 'provisional_not_field_validated',
    'created_on': '2026-06-01',
    'anchor_file': 'data/study_area_anchors_phase1.csv',
    'boundary_file': 'data/interim/study_area_boundary_phase1.geojson',
    'method': 'Convex hull around provisional anchors in EPSG:27700, buffered by 500 metres, exported to EPSG:4326 GeoJSON.',
    'buffer_meters': BOUNDARY_BUFFER_METERS,
    'area_sq_km_approx': round(boundary_bng_area_sq_km, 4),
    'bounds_wgs84': bounds_wgs84,
    'funding_use': 'not_approved',
    'caveat': 'This is a data acquisition boundary only. It is not a design, planning, legal, or funding boundary.',
}

metadata_path = INTERIM / 'study_area_boundary_phase1_metadata.json'
metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
metadata

## Boundary Preview Map

This map is a review aid only. It helps reviewers check whether the provisional boundary looks reasonable before data acquisition starts.

In [ ]:
import folium

MAPS = PROJECT_ROOT / 'outputs' / 'maps'
MAPS.mkdir(parents=True, exist_ok=True)

map_center = [anchors['latitude'].mean(), anchors['longitude'].mean()]
preview_map = folium.Map(location=map_center, zoom_start=15, tiles='OpenStreetMap')
folium.GeoJson(
    json.loads(boundary.to_json()),
    name='Provisional Gate 0 boundary',
    style_function=lambda feature: {
        'fillColor': '#2f80ed',
        'color': '#1d4f91',
        'weight': 2,
        'fillOpacity': 0.18,
    },
).add_to(preview_map)

for row in anchors.itertuples(index=False):
    folium.Marker(
        location=[row.latitude, row.longitude],
        tooltip=f'{row.anchor_name} ({row.anchor_role})',
    ).add_to(preview_map)

map_path = MAPS / 'gate0_study_boundary_preview.html'
preview_map.save(map_path)
map_path

## Priority 1 Acquisition Ledger

The next acquisition step is limited to Priority 1 sources. Do not download later sources until the boundary and acquisition process are reviewed.

In [ ]:
sources = pd.read_csv(DATA / 'source_registry_phase1.csv')
acquisition = pd.read_csv(DATA / 'source_acquisition_status_phase1.csv')

priority_1 = acquisition.loc[acquisition['priority_group'] == 'Priority 1'].merge(
    sources[['source_id', 'source_name', 'owner', 'trust_tier', 'url', 'license']],
    on='source_id',
    how='left',
)

assert priority_1['source_name'].notna().all(), 'Every Priority 1 source must resolve to the source registry.'
priority_1[['source_id', 'source_name', 'owner', 'trust_tier', 'license', 'forensic_status', 'next_action', 'url']]

In [ ]:
claims = pd.read_csv(DATA / 'evidence_claims_register.csv')
claims[['claim_id', 'claim_type', 'minimum_evidence_level', 'current_evidence_level', 'can_use_for_funding', 'next_action']]

## Gate 0 Review Decision

Before acquisition, a reviewer should answer:

- Are the route nodes correct enough for first-pass data acquisition?
- Is a 500 metre buffer appropriate for the first public-data pull?
- Should any anchor be added, removed, or moved?
- Should the first route family be Colmore Row, Moor Street, Aston University, or all of them?

Only after that review should Priority 1 source acquisition begin.